# Notebook 3 — Embeddings & FAISS (Quickstart)

Small runnable example that generates embeddings (sentence-transformers if available) or falls back to TF-IDF, then runs a FAISS search (or simple cosine fallback).

In [ ]:
# Cell 2: Build simple embeddings and run a quick search
import numpy as np

texts = [
    'DNA sequencing determines the order of nucleotides',
    'CRISPR is a gene editing tool',
    'Protein folding depends on amino acid sequence',
    'The dog is sleeping in the park'
]

# Try sentence-transformers first
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(texts)
    print('Used sentence-transformers (all-MiniLM-L6-v2)')
except Exception as e:
    print('sentence-transformers not available or failed — falling back to sklearn TF-IDF')
    from sklearn.feature_extraction.text import TfidfVectorizer
    v = TfidfVectorizer().fit_transform(texts)
    embeddings = v.toarray()

# Try FAISS for fast search (if installed)
try:
    import faiss
    d = embeddings.shape[1]
    index = faiss.IndexFlatL2(d)
    index.add(embeddings.astype('float32'))
    query = 'How does CRISPR work?'
    # create query embedding
    try:
        if 'model' in globals():
            qemb = model.encode([query]).astype('float32')
        else:
            qemb = TfidfVectorizer().fit(texts + [query]).transform([query]).toarray().astype('float32')
    except Exception:
        qemb = np.mean(embeddings, axis=0, keepdims=True).astype('float32')

    D, I = index.search(qemb, k=3)
    print('
Top matches (FAISS):')
    for dist, idx in zip(D[0], I[0]):
        print(f'- {texts[idx]} (distance={dist:.4f})')
except Exception as e:
    print('FAISS not available — performing simple cosine similarity fallback')
    from sklearn.metrics.pairwise import cosine_similarity
    if 'model' in globals():
        qemb = model.encode([query])
    else:
        qemb = np.mean(embeddings, axis=0, keepdims=True)
    sims = cosine_similarity(qemb, embeddings)[0]
    order = np.argsort(-sims)[:3]
    print('
Top matches (cosine fallback):')
    for idx in order:
        print(f'- {texts[idx]} (score={sims[idx]:.4f})')

## Notes
- For real workshops, install `sentence-transformers` and `faiss-cpu` (or `faiss-gpu`).
- Use domain-specific embedding models for biology when possible (PubMedBERT, DNABert, ESM2 for proteins).